# Differential gene expression

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths
import gc
import time

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# dream
import dreampy as dp

# Parallel processing
from joblib import Parallel, delayed, parallel_backend


# dataframes
import pandas as pd
import numpy as np
from collections import defaultdict
# plotting
import matplotlib.pyplot as plt 

# Diff genes
from flash_mm import lmmfit, lmmtest, contrast_matrix, sslmm, lmm
from statsmodels.stats.multitest import multipletests

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
base_dir = str(here('data/differential_abundance/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
de_dir = os.path.join(base_dir, 'de_analysis_groups') 
de_overall_dir = os.path.join(base_dir, 'de_analysis_overall') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

In [3]:
mdata =  md.read(os.path.join(objects_dir, "mdata_666_annotation.h5mu"), backed='r')

## Differential gene expression

#### Map cells to each neighborhood

In [4]:
adata            = mdata["rna"]
nhoods           = adata.obsm["nhoods"]                          
ref_cells        = adata.obs_names[adata.obs["nhood_ixs_refined"] == 1]

rows, cols = nhoods.nonzero()
df = pd.DataFrame({
    "cell"       : adata.obs_names[rows],
    "nhood_id"   : cols,
    "index_cell" : ref_cells[cols],
})

#### Combine differential abundance and cell data

In [ ]:
# ----------------------------- SET UP -----------------------------------
print("Setting up variables")
prefix           = "pre_vs_nd"
out_dir          = de_overall_dir 
da_results_path  = os.path.join(files_dir, f"{prefix}.csv")

nhood_anno_key   = "nhood_annotation"
sample_key       = "ic_id_platform_adjusted_sample"
donor_key        = "ic_id_donor_overall"
disease_key      = "disease_harmonized"
comp             = "direction"
# ----------------------------- LOAD -------------------------------------
da = pd.read_csv(da_results_path, index_col=0)
# ----------------------------- PREPROCESS -------------------------------
print("Combine differential abundance data with per cell neighborhoods")
df_t2d = df.merge(da, how="left")
df_t2d ["direction"] = "no_change"

# Direction of regulations
df_t2d.loc[(df_t2d["SpatialFDR"] <= 0.1) & (df_t2d["logFC"] > 0), "direction"] = "up"
df_t2d.loc[(df_t2d["SpatialFDR"] <= 0.1) & (df_t2d["logFC"] < 0), "direction"] = "down"
df_t2d ["anno_direction"] = df_t2d[nhood_anno_key] + "_" + df_t2d["direction"]

df_t2d = df_t2d .set_index('cell')

Setting up variables
Combine differential abundance data with per cell neighborhoods


In [ ]:
for anno_id in list(df_t2d['nhood_annotation'].drop_duplicates()):
    print(f"\nProcessing: {anno_id}")

    # Subset to annotation (neighborhoods)
    df_sub = (df_t2d.loc[df_t2d[nhood_anno_key] == anno_id, [nhood_anno_key, comp]]
               .loc[lambda x: ~x.index.duplicated(keep="first")])
    
    subset = adata[adata.obs_names.isin(df_sub.index)].to_memory()
    subset.obs = subset.obs.join(
        df_sub.reindex(subset.obs.index)
    )
    
    print(f"    Total cells: {len(subset)}")
    
    # Pseudobulk aggregation (by comparison group + sample)
    pb = dp.aggregate_pseudobulk(
        subset,
        layer='counts',
        groupby=[nhood_anno_key, sample_key, comp]
    )
    
    # Preprocessing
    try:
        pb = dp.filter_samples(pb, min_cells=50, min_samples=3)
    except ValueError as e:
        print(f"    Skipping {anno_id}: {e}")
        continue
        
    pb = dp.compute_tmm_factors(pb, assay_col="assays")
    # Add log2(total cells per sample) from n_cells column
    if 'n_cells' in pb.obs.columns:
        pb.obs['log2_n_cells'] = np.log2(pb.obs['n_cells'] + 1)
    assays = dp.filter_by_expr(pb, assay_col="assays")

    # Ensure random effect columns exist
    for col in ['library_prep', 'ic_id_donor_overall']:
        if col not in pb.obs.columns:
            print(f"    Warning: {col} not found in metadata")
            pb.obs[col] = 'unknown'
    
    # Define formula 
    formula = "~ {} + log2_n_cells + (1|library_prep) + (1|ic_id_donor_overall)".format(comp)
    
    print(f"    Formula: {formula}") 
    
    for assay_name, assay_pb in assays.items():
        print(f"    Processing assay: {assay_name}")
        
        comp_col = comp
    
        # ------------------ CHECK + PREP FACTOR ------------------
        if comp_col not in assay_pb.obs.columns:
            print(f"      Skipping: {comp_col} not found")
            continue
    
        assay_pb.obs[comp_col] = assay_pb.obs[comp_col].astype("category")
        levels = list(assay_pb.obs[comp_col].cat.categories)
    
        print(f"      Levels: {levels}")
    
        # Skip if ≤1 group
        if len(levels) <= 1:
            print("      Skipping: only one group present")
            continue
    
        # ------------------ SELECT COMPARISON ------------------
        if {"up", "down"}.issubset(levels):
            ref = "down"
            target_coef = f"{comp_col}_up"
            print("      Comparison: up vs down")
    
        elif {"up", "no_change"}.issubset(levels):
            ref = "no_change"
            target_coef = f"{comp_col}_up"
            print("      Comparison: up vs no_change")
    
        elif {"down", "no_change"}.issubset(levels):
            ref = "no_change"
            target_coef = f"{comp_col}_down"
            print("      Comparison: down vs no_change")
    
        else:
            print("      Skipping: no valid comparison pair")
            continue
    
        # ------------------ ENFORCE REFERENCE ------------------
        desired_order = ["down", "no_change", "up"]
        present_order = [lvl for lvl in desired_order if lvl in levels]
    
        # put chosen reference first
        new_order = [ref] + [lvl for lvl in present_order if lvl != ref]
    
        assay_pb.obs[comp_col] = assay_pb.obs[comp_col].cat.reorder_categories(
            new_order,
            ordered=True
        )
    
        print(f"      Reference level: {ref}")
        print(f"      Category order: {list(assay_pb.obs[comp_col].cat.categories)}")
    
        # ------------------ MODEL PIPELINE ------------------
        assay_pb = dp.log2cpm(assay_pb)
    
        try:
            assay_pb = dp.estimate_weights(assay_pb, formula=formula, n_jobs = 60)
        except Exception as e:
            print(f"      Error estimating weights: {e}")
            continue
    
        try:
            fit = dp.fit_models(assay_pb, formula=formula, n_jobs = 60)
        except Exception as e:
            print(f"      Error fitting models: {e}")
            continue
    
        fit_eb = dp.ebayes(fit)
    
        # ------------------ VALIDATE COEFFICIENT ------------------
        coef_names = fit_eb.coef_names
        print(f"      Coefficients: {coef_names}")
    
        if target_coef not in coef_names:
            print(f"      Skipping: expected coef '{target_coef}' not found")
            continue
    
        # Extra safety: ensure reference really worked
        ref_coef_name = f"{comp_col}_{ref}"
        assert ref_coef_name not in coef_names, \
            f"Reference level '{ref}' incorrectly encoded"
    
        # ------------------ EXTRACT RESULTS ------------------
        test_results = dp.get_results(
            fit_eb,
            coef=target_coef,
            assay_name=assay_name,
            p_value=0.05,
            lfc=1.0
        )
    
        test_results.to_csv(os.path.join(files_dir, f"dreampy_{prefix}_{anno_id}_{target_coef}.csv"), index=False)